

ANN Classification



Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN)

This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

Start Simple: Begin with a simple architecture and gradually increase complexity if needed. Grid Search/Random Search: Use grid search or random search to try different architectures.

• Cross-Validation: Use cross-validation to evaluate the performance of different architectures.

Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as: • The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.

A common practice is to start with 1-2 hidden layers.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
data=pd.read_csv('Churn_Modelling.csv')

In [3]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
one_hot_encoder=OneHotEncoder()
geo_encoder=one_hot_encoder.fit_transform(data[['Geography']])

geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=one_hot_encoder.get_feature_names_out(['Geography']))


data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)


X=data.drop('Exited',axis=1)
y=data['Exited']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## Scale these features

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [4]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
    
    with open('one_hot_encoder.pkl','wb') as file:
        pickle.dump(one_hot_encoder,file)
        
        with open('scaler.pkl','wb') as file:
          pickle.dump(scaler, file)

In [5]:
## Define a function to create the model and try diffrent paramenters(kerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    
    for _  in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
        
        
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])
        
    return model

In [6]:
model = create_model(neurons=32, layers=2)
model.summary()

c:\Users\DELL\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,505 (5.88 KB)

 Trainable params: 1,505 (5.88 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
## Create a Keras Classsifier
model=KerasClassifier(layers=1,neurons=32,model=create_model,epochs=30,batch_size=10,verbose=0)

In [8]:
# Define the grid search parameters

param_grid={
    'neurons' : [16,32],
    'layers' : [1,2],
    'epochs' : [15,30]
    
}

In [9]:
# Perform grid search

grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3,verbose=1)
grid_result=grid.fit(X_train,y_train)

# Print the best parameters
print("Best:%f using %s" %(grid_result.best_score_,grid_result.best_params_))

Fitting 3 folds for each of 8 candidates, totalling 24 fits


c:\Users\DELL\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best:0.857249 using {'epochs': 15, 'layers': 2, 'neurons': 16}
